# Load RESPOND News Data

This notebook reads the raw country news CSV files from the WebDAV project folder, such as `Bulgaria_news.csv`, `Italy_news.csv`, `Netherlands_news.csv`, and `United_Kingdom_news.csv`.

It can also load existing annotation output files from the `annotations/` folder if you want to inspect prior classification results.

In [ ]:
from pathlib import Path
import os

import pandas as pd

from config import NEWS_FOLDER, SELECTED_COUNTRIES
from dataloader import load_country_news_files

In [ ]:
# Data location shown in the screenshot:
# ~/webdav/ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/
#
# To use a different mount location, set RESPOND_NEWS_FOLDER before running this notebook.

NEWS_DIR = Path(os.environ.get("RESPOND_NEWS_FOLDER", NEWS_FOLDER)).expanduser()
COUNTRIES = SELECTED_COUNTRIES

print(f"News directory: {NEWS_DIR}")
print(f"Countries:      {', '.join(COUNTRIES)}")

In [ ]:
if not NEWS_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {NEWS_DIR}. Mount the WebDAV folder at this location, "
        "or set RESPOND_NEWS_FOLDER to the folder containing the *_news.csv files."
    )

expected_files = [NEWS_DIR / f"{country}_news.csv" for country in COUNTRIES]
missing_files = [path.name for path in expected_files if not path.exists()]

if missing_files:
    available_files = sorted(NEWS_DIR.glob("*.csv"))
    preview = "\n".join(f"- {path.name}" for path in available_files[:25])
    message = (
        "Some expected country news files are missing:\n"
        + "\n".join(f"- {name}" for name in missing_files)
    )
    if preview:
        message += f"\n\nCSV files currently in NEWS_DIR:\n{preview}"
    raise FileNotFoundError(message)

df = load_country_news_files(news_folder=str(NEWS_DIR), countries=COUNTRIES)
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head()

In [ ]:
df.info()

In [ ]:
# Quick checks that are useful for the raw news data.
for column in ["country", "source.uri", "lang", "dateTime"]:
    if column in df.columns:
        display(df[column].value_counts(dropna=False).rename_axis(column).to_frame("count"))

## Optional: Load Annotation Data

Use this when you want the original validation annotations from `config.ANNOTATION_FILE`.

In [ ]:
from dataloader import load_human_annotated_for_translation

df_annotations = load_human_annotated_for_translation()
print(f"Loaded {len(df_annotations):,} annotated rows.")
df_annotations.head()